# N-CMAPSS DS05 — is the gap a data-budget artifact?

## The observation that motivates this

We do **not** fail to replicate everywhere. On C-MAPSS FD003 we are 2nd of 9, ahead of five
published methods **including MSTSDN**. On the battery our best-seed range overlaps the published
range. N-CMAPSS is the only dataset where we are behind everything comparable.

The decisive clue is that the *same* method points both ways:

| | trivial floor | ours | MSTSDN | our skill | their skill |
|---|---|---|---|---|---|
| C-MAPSS FD003 | 42.17 | 10.398 | 13.66 | **+0.753** | +0.676 |
| N-CMAPSS DS05 | 23.36 | 8.247 | 3.95 | +0.647 | **+0.831** |

If their architecture were simply better, they would beat us on both. So something in our
N-CMAPSS pipeline is specific — and there is an obvious candidate.

## We train on 6.7% of the available data

| | rows |
|---|---|
| DS05 development set | 4,350,606 |
| after holding out units 5 and 6 for validation | 2,900,404 |
| after subsampling 1-in-10 | **290,040** |

Published works use all six development units and describe no subsampling. That is roughly a 15x
difference in training data before any architecture question arises.

## The experiment

A **learning curve**: hold the optimization budget fixed at a set number of gradient updates and
vary only the amount of data. Fixing updates rather than epochs is what separates "more data" from
"more training" — with a fixed epoch count, larger datasets silently receive more optimization too.

- If RMSE keeps falling as data grows and has not flattened at 100%, we are data-limited and the
  gap is an artifact of our budget.
- If it plateaus near 8.2, more data will not help and the difference is genuinely architectural.

Part 2 then varies the number of **training units** (4, 5, 6) at fixed data volume, which separates
sheer quantity from engine diversity.

In [ ]:
import os, json, gc, time, random, math
os.environ['PYTHONHASHSEED']='42'
import numpy as np, pandas as pd, h5py
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g,True)
    except RuntimeError as _e: print('memory growth:',_e)
from tensorflow.keras import backend as K, regularizers
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Concatenate, GlobalAveragePooling1D, GlobalMaxPooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
def set_seeds(s):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s)
    tf.keras.utils.set_random_seed(s)
def preflight():
    if not tf.config.list_physical_devices('GPU'): print('CPU mode'); return False
    try:
        x=np.random.rand(8,10,3).astype('float32'); y=np.random.rand(8,1).astype('float32')
        i=Input(shape=(10,3)); o=Dense(1)(GlobalAveragePooling1D()(Conv1D(4,3,padding='same')(i)))
        m=Model(i,o); m.compile('adam','mse'); m.fit(x,y,epochs=1,verbose=0); K.clear_session()
        print('GPU preflight passed'); return True
    except Exception as e:
        print('GPU preflight FAILED (%s)'%type(e).__name__); K.clear_session(); return False
USE_GPU=preflight(); DEVICE='/GPU:0' if USE_GPU else '/CPU:0'
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('determinism:',e)
print('TF',tf.__version__,'| device',DEVICE)

In [ ]:
SEED=42; BATCH=512; UPDATES=60_000; N_SEEDS=3
SNAP_V3=8.247            # our current DS05 result, 1-in-10 subsample, 4 training units
set_seeds(SEED)

def read_sub(dset,step,dtype=np.float32,chunk=400_000):
    n=dset.shape[0]; out=[]
    for a in range(0,n,chunk):
        b=min(a+chunk,n); blk=dset[a:b]; first=(-a)%step
        if first<len(blk): out.append(blk[first::step].astype(dtype,copy=False))
    return np.concatenate(out)

BASE_S=2                  # load at 1-in-2, then thin further in memory
with h5py.File('N-CMAPSS_DS05.h5','r') as h:
    Wd=read_sub(h['W_dev'],BASE_S);  Xd=read_sub(h['X_s_dev'],BASE_S)
    Ad=read_sub(h['A_dev'],BASE_S);  Yd=read_sub(h['Y_dev'],BASE_S)
    Wt=read_sub(h['W_test'],10);     Xt=read_sub(h['X_s_test'],10)
    At=read_sub(h['A_test'],10);     Yt=read_sub(h['Y_test'],10)
FD=np.hstack([Wd,Xd]).astype(np.float32); ud=Ad[:,0].astype(int); yd=Yd.ravel().astype(np.float32)
FT=np.hstack([Wt,Xt]).astype(np.float32); ut=At[:,0].astype(int); yt=Yt.ravel().astype(np.float32)
del Wd,Xd,Ad,Yd,Wt,Xt,At,Yt; gc.collect()
NF=FD.shape[1]
print('loaded dev at 1-in-%d: %s rows (%.0f MB) | test at 1-in-10: %s rows'
      %(BASE_S,format(len(yd),','),FD.nbytes/1e6,format(len(yt),',')))

def make(train_units,val_units,thin):
    """thin is relative to the 1-in-2 load, so effective subsample = BASE_S * thin."""
    tr=np.isin(ud,train_units); va=np.isin(ud,val_units)
    itr=np.flatnonzero(tr)[::thin]; iva=np.flatnonzero(va)[::max(1,thin)]
    sc=MinMaxScaler().fit(FD[itr])
    f=lambda X: sc.transform(X).reshape(-1,NF,1).astype(np.float32)
    return dict(Xtr=f(FD[itr]),ytr=yd[itr],Xva=f(FD[iva]),yva=yd[iva],
                Xte=f(FT),yte=yt,n=len(itr))

def build(seed=SEED,filters=(96,128,128,32),lstm_units=80,dense=(192,96),
          dropout=0.0,l2reg=0.0,lr=1e-3):
    kw=dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed),
            kernel_regularizer=regularizers.l2(l2reg) if l2reg>0 else None)
    inp=Input(shape=(NF,1)); c=inp
    for i,f in enumerate(filters):
        c=Conv1D(f,3,padding='same',activation='relu',**kw)(c)
        if dropout>0: c=Dropout(dropout,seed=seed)(c)
        if i<2 and int(c.shape[1])>2: c=MaxPooling1D(2,padding='same')(c)
    cv=GlobalAveragePooling1D()(c)
    r=LSTM(lstm_units,**kw)(inp)
    h=Concatenate()([cv,r])
    for d in dense: h=Dense(d,activation='relu',**kw)(h)
    m=Model(inp,Dense(1,activation='linear',**kw)(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),loss='mse'); return m

def run(d,seed,updates=UPDATES):
    """Fixed number of gradient updates, so data volume is the only thing varying."""
    steps=max(1,len(d['ytr'])//BATCH)
    epochs=max(1,int(math.ceil(updates/steps)))
    K.clear_session(); gc.collect(); set_seeds(seed); t0=time.time()
    with tf.device(DEVICE):
        m=build(seed=seed)
        hh=m.fit(d['Xtr'],d['ytr'],validation_data=(d['Xva'],d['yva']),epochs=epochs,
                 batch_size=BATCH,verbose=0,
                 callbacks=[EarlyStopping('val_loss',patience=max(5,epochs//6),restore_best_weights=True),
                            ReduceLROnPlateau('val_loss',factor=0.5,patience=max(3,epochs//12),min_lr=1e-6)])
        rm=lambda X,y: float(np.sqrt(mean_squared_error(y,m.predict(X,batch_size=8192,verbose=0))))
        v=rm(d['Xva'],d['yva']); t=rm(d['Xte'],d['yte'])
    ep=len(hh.history['loss'])
    return dict(val=v,test=t,epochs=ep,steps=steps,updates=ep*steps,secs=round(time.time()-t0))

## Part 1 — learning curve at fixed optimization budget

Training units 1-4, validation units 5-6 throughout, so only the data volume changes.

In [ ]:
TRAIN_U=[1,2,3,4]; VAL_U=[5,6]
LEVELS=[(5,'1-in-10  (current)'),(2,'1-in-4'),(1,'1-in-2 (all loaded)')]
R=[]; t0=time.time()
for thin,label in LEVELS:
    d=make(TRAIN_U,VAL_U,thin)
    rs=[run(d,SEED+s) for s in range(N_SEEDS)]
    te=np.array([r['test'] for r in rs]); va=np.array([r['val'] for r in rs])
    R.append(dict(level=label,n_train=d['n'],test=te.mean(),test_sd=te.std(ddof=1),val=va.mean(),
                  updates=np.mean([r['updates'] for r in rs]),secs=np.mean([r['secs'] for r in rs])))
    print('%-20s n=%9s | test %6.3f +/- %5.3f | val %6.3f | %6.0f updates | %4.0fs/run'
          %(label,format(d['n'],','),te.mean(),te.std(ddof=1),va.mean(),
            np.mean([r['updates'] for r in rs]),np.mean([r['secs'] for r in rs])))
    del d; gc.collect()
P=pd.DataFrame(R); pd.set_option('display.width',200)
print('\n'+P.to_string(index=False))
print('\nv3 reference at 1-in-10 with a different budget: %.3f'%SNAP_V3)
if len(P)>1:
    slope=(P.test.iloc[-1]-P.test.iloc[0])/math.log10(P.n_train.iloc[-1]/P.n_train.iloc[0])
    print('slope: %+.3f RMSE per 10x of data'%slope)
    print('VERDICT: %s'%('still improving - we are DATA-LIMITED'
                         if P.test.iloc[-1]<P.test.iloc[0]-P.test_sd.mean()
                         else 'flat - more data will not close the gap'))
print('part 1 finished in %.0f s'%(time.time()-t0))

## Part 2 — how many engines, rather than how many rows

Same number of training rows in each case, drawn from a different number of engines. This separates
data volume from engine diversity: validation shrinks to fewer held-out units as training grows.

In [ ]:
TARGET_N=P.n_train.max()
CONFIGS=[([1,2,3,4],[5,6],'4 units (current)'),
         ([1,2,3,4,5],[6],'5 units'),
         ([1,2,3,4,6],[5],'5 units (alt)')]
R2=[]
for tu,vu,label in CONFIGS:
    d=make(tu,vu,1)
    if d['n']>TARGET_N:                        # equalise row count across configurations
        idx=np.linspace(0,d['n']-1,TARGET_N).astype(int)
        d['Xtr']=d['Xtr'][idx]; d['ytr']=d['ytr'][idx]; d['n']=TARGET_N
    rs=[run(d,SEED+s) for s in range(N_SEEDS)]
    te=np.array([r['test'] for r in rs]); va=np.array([r['val'] for r in rs])
    R2.append(dict(config=label,units=len(tu),n_train=d['n'],test=te.mean(),
                   test_sd=te.std(ddof=1),val=va.mean()))
    print('%-20s %d units | n=%9s | test %6.3f +/- %5.3f | val %6.3f'
          %(label,len(tu),format(d['n'],','),te.mean(),te.std(ddof=1),va.mean()))
    del d; gc.collect()
P2=pd.DataFrame(R2)
print('\n'+P2.to_string(index=False))
d4=P2[P2.units==4].test.iloc[0]; d5=P2[P2.units==5].test.mean()
print('\n4 units %.3f vs 5 units %.3f -> %+.3f from one extra engine at equal row count'%(d4,d5,d5-d4))

## Summary

In [ ]:
best_n=P.nsmallest(1,'test').iloc[0]; best_u=P2.nsmallest(1,'test').iloc[0]
print('='*74)
print('%-40s %8s %8s'%('','test','n_train'))
print('%-40s %8.3f %8s'%('v3 published pipeline (1-in-10, 4 units)',SNAP_V3,format(P.n_train.iloc[0],',')))
print('%-40s %8.3f %8s'%('best data budget in Part 1',best_n.test,format(int(best_n.n_train),',')))
print('%-40s %8.3f %8s'%('best unit configuration in Part 2',best_u.test,format(int(best_u.n_train),',')))
print('%-40s %8.3f'%('MSTSDN (best comparable published)',3.95))
print('%-40s %8.3f'%('trivial floor',23.362))
print('='*74)
gain=SNAP_V3-min(best_n.test,best_u.test)
print('\ndata budget alone accounts for %+.3f RMSE'%(-gain))
print('remaining gap to MSTSDN: %+.3f'%(min(best_n.test,best_u.test)-3.95))
print('\nNOTE: full data is 1-in-1 across all 6 dev units = 4,350,606 rows.')
print('This notebook reaches 1-in-2 on 4-5 units. If the curve is still descending at that')
print('point, the next step is the full set with a proportionally larger update budget.')
P.to_csv('ncmapss_learning_curve.csv',index=False); P2.to_csv('ncmapss_unit_sweep.csv',index=False)
json.dump(dict(levels=R,units=R2,v3=SNAP_V3,trivial=23.362),
          open('ncmapss_data_budget.json','w'),indent=2,default=str)
print('\nsaved ncmapss_learning_curve.csv, ncmapss_unit_sweep.csv, ncmapss_data_budget.json')